# Caption LAION-Art with LLaVA

Generate concise LLaVA-1.5-7B captions for every clean LAION-Art image. Results are checkpointed after each batch, so rerunning resumes from the existing CSV.


In [1]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration

DATA_DIR = Path("dataset/laion_art")
CLEAN_DIR = DATA_DIR / "clean"
OUTPUT_PATH = DATA_DIR / "llava_captions.csv"
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
PROMPT = "Describe the image in twenty words or less."
BATCH_SIZE = 16
MAX_NEW_TOKENS = 77
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for LLaVA captioning.")
device = "cuda"
dtype = torch.float16


In [2]:
# Reuse the repository's LLaVA-1.5-7B setup.
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
).to(device).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
# Resume from non-empty captions already saved for clean image IDs.
image_paths = sorted(
    path for path in CLEAN_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)
if not image_paths:
    raise FileNotFoundError(f"No images found in {CLEAN_DIR}")
if len({path.stem for path in image_paths}) != len(image_paths):
    raise ValueError("Clean image filenames must have unique stems.")

columns = ["image_id", "image_path", "llava_caption"]
if OUTPUT_PATH.exists():
    captions = pd.read_csv(OUTPUT_PATH, dtype={"image_id": str})
    missing_columns = set(columns) - set(captions.columns)
    if missing_columns:
        raise ValueError(f"Missing columns in {OUTPUT_PATH}: {sorted(missing_columns)}")
    if captions["image_id"].duplicated().any():
        raise ValueError(f"Duplicate image IDs in {OUTPUT_PATH}")
    captions = captions[columns].copy()
else:
    captions = pd.DataFrame(columns=columns)

clean_ids = {path.stem for path in image_paths}
captions = captions[captions["image_id"].isin(clean_ids)].copy()
completed_ids = set(
    captions.loc[captions["llava_caption"].fillna("").str.strip().ne(""), "image_id"]
)
pending = [path for path in image_paths if path.stem not in completed_ids]
print(f"Clean images: {len(image_paths):,}; remaining: {len(pending):,}")


Clean images: 59,277; remaining: 59,277


In [4]:
def save_checkpoint(frame):
    frame = frame.drop_duplicates("image_id", keep="last").sort_values("image_id")
    temporary_path = OUTPUT_PATH.with_suffix(".csv.tmp")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(OUTPUT_PATH)
    return frame

prompt = f"USER: <image>\n{PROMPT}\nASSISTANT:"
for start in tqdm(range(0, len(pending), BATCH_SIZE), desc="Captioning batches"):
    batch_paths = pending[start:start + BATCH_SIZE]
    images = []
    for path in batch_paths:
        with Image.open(path) as image:
            images.append(image.convert("RGB"))

    inputs = processor(
        text=[prompt] * len(images),
        images=images,
        padding=True,
        return_tensors="pt",
    )
    inputs = {
        key: value.to(device=device, dtype=dtype) if value.is_floating_point() else value.to(device)
        for key, value in inputs.items()
    }
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    generated = generated[:, inputs["input_ids"].shape[1]:]
    batch_captions = [caption.strip() for caption in processor.batch_decode(generated, skip_special_tokens=True)]

    batch_frame = pd.DataFrame({
        "image_id": [path.stem for path in batch_paths],
        "image_path": [path.as_posix() for path in batch_paths],
        "llava_caption": batch_captions,
    })
    captions = save_checkpoint(pd.concat([captions, batch_frame], ignore_index=True))

expected_ids = {path.stem for path in image_paths}
captioned_ids = set(captions.loc[captions["llava_caption"].fillna("").str.strip().ne(""), "image_id"])
assert expected_ids <= captioned_ids, f"Missing captions for {len(expected_ids - captioned_ids)} images"
print(f"Saved {len(expected_ids):,} captions to {OUTPUT_PATH}")
captions.head()


Captioning batches:   0%|          | 0/3705 [00:00<?, ?it/s]

Saved 59,277 captions to dataset/laion_art/llava_captions.csv


,image_id,image_path,llava_caption
0,000000000000,dataset/laion_art/clean/000000000000.png,A red bridge over a river.
1,000000000001,dataset/laion_art/clean/000000000001.png,A young boy standing in front of a wooden stru...
2,000000000004,dataset/laion_art/clean/000000000004.png,A plate of food with a bowl of salsa.
3,000000000006,dataset/laion_art/clean/000000000006.png,A pink vintage car with a wooden panel on the ...
4,000000000007,dataset/laion_art/clean/000000000007.png,A snowy scene with a large brown bear holding ...
